# Price Model — Feature Search\n",
    "\n",
    "**Target:** annual crop-year USD/kg log-return (`ret_usd`)  \n",
    "**Anchor:** `shortfall_trail` — trailing 5yr production shortfall (baseline R²=0.529 on full n=22)  \n",
    "**Candidates:** equity/FX/commodity annual log-returns  \n",
    "**Excluded from joint models:** `tur_usd` (TUR ETF launched 2008), `ulker_try` (TRY-denominated, endogenous), `cocoa_usd`/`dba_usd`/`tryusd` (n<18) — shown in bivariate only\n",
    "\n",
    "**Joint model sample:** `nestle_usd`, `barry_chf`, `hershey_usd`, `bunge_usd`, `mdlz_usd` + anchor — n=21

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from sklearn.linear_model import LassoCV, RidgeCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
DATA = Path('../data/raw')

In [ ]:

# --- Annual price panel ---
master = pd.read_csv(DATA / 'hazelnut_35yr_master.csv', index_col=0)
master.index = master.index.astype(int)
master['trend_trail5']    = master['prod_mt'].rolling(5, min_periods=3).mean().shift(1)
master['shortfall_trail'] = (master['prod_mt'] - master['trend_trail5']) / master['trend_trail5'] * 100

cy = pd.read_csv(DATA / 'giresun_spot_prices_cropyear.csv').set_index('crop_year')
cy['ret_usd'] = np.log(cy['vwap_usd_kg_inshell']).diff()
annual = cy[['ret_usd']].join(master[['shortfall_trail']], how='left')

# --- Asset basket ---
basket = pd.read_csv(DATA / 'basket/equity_basket_prices.csv', index_col=0)
basket.index = basket.index.astype(int)
ALL_ASSETS = [c for c in basket.columns if c != 'tur_usd']
ret_basket = np.log(basket[ALL_ASSETS] / basket[ALL_ASSETS].shift(1))
ret_basket.index.name = 'crop_year'

df = annual.join(ret_basket, how='left')

# Joint model: exclude ulker (TRY-denominated/endogenous) and sparse features
DENSE = ['shortfall_trail', 'nestle_usd', 'barry_chf', 'hershey_usd', 'bunge_usd', 'mdlz_usd']
JOINT_EXCLUDE = ['tur_usd', 'ulker_try', 'cocoa_usd', 'dba_usd', 'tryusd']

print(f'Full panel: {df["ret_usd"].notna().sum()} crop years')
print(f'Pairwise coverage:')
print(df[['shortfall_trail'] + ALL_ASSETS].notna().sum().sort_values())
print(f'\nJoint model complete cases: n={df[["ret_usd"] + DENSE].dropna().shape[0]}')

## Bivariate correlations — all features

In [ ]:
rows = []
for col in ['shortfall_trail'] + ALL_ASSETS:
    sub = df[['ret_usd', col]].dropna()
    if len(sub) < 8 or sub[col].std() == 0: continue
    r, p = stats.pearsonr(sub['ret_usd'], sub[col])
    if col in JOINT_EXCLUDE:
        note = 'excluded — endogenous' if col == 'ulker_try' else 'excluded — sparse'
    elif col == 'shortfall_trail':
        note = 'anchor'
    else:
        note = 'in joint model'
    rows.append({'feature': col, 'r': round(r, 3), 'R²': round(r**2, 3),
                 'p': round(p, 3), 'n': len(sub), 'joint model': note})

corr_df = pd.DataFrame(rows).sort_values('R²', ascending=False).set_index('feature')
corr_df

## OLS — dense feature set (n=21)

In [ ]:
d = df[['ret_usd'] + DENSE].dropna()
print(f'n={len(d)}  k={len(DENSE)}  (n/k={len(d)/len(DENSE):.1f})')
m_ols = sm.OLS(d['ret_usd'], sm.add_constant(d[DENSE])).fit()
print(m_ols.summary2())

## Ridge (L2)

In [ ]:
y = d['ret_usd'].values
X = d[DENSE].values
n, cv = len(y), min(5, len(y) // 4)

scaler = StandardScaler()
Xs = scaler.fit_transform(X)

ridge = RidgeCV(alphas=np.logspace(-3, 4, 300), cv=cv)
ridge.fit(Xs, y)
yhat_r  = ridge.predict(Xs)
r2_r    = 1 - np.sum((y - yhat_r)**2) / np.sum((y - y.mean())**2)
cv_r2_r = cross_val_score(ridge, Xs, y, cv=cv, scoring='r2').mean()

print(f'Ridge  α={ridge.alpha_:.4f}  R²(IS)={r2_r:.3f}  CV R²={cv_r2_r:.3f}  n={n}')
coef_r = pd.DataFrame({'feature': DENSE, 'coef (std)': ridge.coef_.round(4)}).set_index('feature')
coef_r.reindex(coef_r['coef (std)'].abs().sort_values(ascending=False).index)

## Relaxed Lasso→Ridge (L1→L2)

In [ ]:
lasso = LassoCV(alphas=np.logspace(-4, 2, 300), cv=cv, max_iter=50000, n_jobs=-1)
lasso.fit(Xs, y)
mask = np.abs(lasso.coef_) > 1e-6

if mask.sum() == 0:
    lasso.alpha_ = lasso.alphas_[len(lasso.alphas_) // 4]
    lasso.coef_  = Lasso(alpha=lasso.alpha_, max_iter=50000).fit(Xs, y).coef_
    mask = np.abs(lasso.coef_) > 1e-6

sel = [DENSE[i] for i in range(len(DENSE)) if mask[i]]
print(f'Lasso α={lasso.alpha_:.5f}  selected {mask.sum()}/{len(DENSE)}: {sel}')

ridge_lr = RidgeCV(alphas=np.logspace(-3, 4, 200), cv=cv)
ridge_lr.fit(Xs[:, mask], y)
yhat_lr  = ridge_lr.predict(Xs[:, mask])
r2_lr    = 1 - np.sum((y - yhat_lr)**2) / np.sum((y - y.mean())**2)
cv_r2_lr = cross_val_score(ridge_lr, Xs[:, mask], y, cv=cv, scoring='r2').mean()

print(f'Ridge refit  α={ridge_lr.alpha_:.4f}  R²(IS)={r2_lr:.3f}  CV R²={cv_r2_lr:.3f}')
coef_lr = pd.DataFrame({'feature': sel, 'coef (std)': ridge_lr.coef_.round(4)}).set_index('feature')
coef_lr

## Comparison

In [ ]:
m_base = sm.OLS(d['ret_usd'], sm.add_constant(d[['shortfall_trail']])).fit()

cmp = pd.DataFrame([
    {'': 'Baseline', 'Estimator': 'OLS', 'Features': 'shortfall_trail only',
     'k': 1, 'n': int(m_base.nobs),
     'R²(IS)': round(m_base.rsquared, 3), 'AIC': round(m_base.aic, 1), 'CV R²': '—'},
    {'': 'OLS', 'Estimator': 'OLS', 'Features': 'all dense',
     'k': len(DENSE), 'n': len(d),
     'R²(IS)': round(m_ols.rsquared, 3), 'AIC': round(m_ols.aic, 1), 'CV R²': '—'},
    {'': 'Ridge', 'Estimator': 'L2', 'Features': 'all dense (shrunk)',
     'k': len(DENSE), 'n': n,
     'R²(IS)': round(r2_r, 3), 'AIC': '—', 'CV R²': round(cv_r2_r, 3)},
    {'': 'Lasso→Ridge', 'Estimator': 'L1→L2', 'Features': ', '.join(sel),
     'k': len(sel), 'n': n,
     'R²(IS)': round(r2_lr, 3), 'AIC': '—', 'CV R²': round(cv_r2_lr, 3)},
]).set_index('')
cmp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: coefficient chart — Ridge vs Lasso→Ridge
ax = axes[0]
x = np.arange(len(DENSE))
w = 0.35
ax.bar(x - w/2, coef_r.reindex(DENSE)['coef (std)'], width=w,
       label='Ridge', color='steelblue', alpha=0.8)
lasso_vals = pd.Series(dict(zip(sel, ridge_lr.coef_))).reindex(DENSE).fillna(0)
ax.bar(x + w/2, lasso_vals, width=w,
       label='Lasso→Ridge', color='#e74c3c', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(DENSE, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Coefficient (std. units)')
ax.set_title('Ridge vs Lasso→Ridge coefficients')
ax.legend(fontsize=9)

# Right: actual vs fitted (Lasso→Ridge)
ax = axes[1]
ax.scatter(yhat_lr * 100, y * 100, s=45, c='steelblue', zorder=3)
for yr, yh, ya in zip(d.index, yhat_lr * 100, y * 100):
    if abs(ya - yh) > 15:
        ax.annotate(str(yr), (yh, ya), xytext=(4, 3),
                    textcoords='offset points', fontsize=7)
mn = min(yhat_lr.min(), y.min()) * 100 - 3
mx = max(yhat_lr.max(), y.max()) * 100 + 3
ax.plot([mn, mx], [mn, mx], 'k--', lw=0.8)
ax.set_xlabel('Fitted (%)')
ax.set_ylabel('Actual (%)')
ax.set_title(f'Lasso→Ridge: actual vs fitted  R²={r2_lr:.3f}  CV R²={cv_r2_lr:.3f}')

plt.tight_layout()
plt.show()